### Procesamiento de Lenguaje Natural I
# **Desafío 1**
## Alumno: Martin Madrid
## n°SIU: a2409


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [ ]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [ ]:
print(newsgroups_train.data[0])

Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [ ]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [ ]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [ ]:
tfidfvect.vocabulary_['car']

Probamos con una palbra que no está en el documento.

In [ ]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [ ]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [ ]:
y_train = newsgroups_train.target
y_train[:10]

Hay 20 clases correspondientes a los 20 grupos de noticias

In [ ]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [ ]:
idx = 4811
print(newsgroups_train.data[idx])

Medimos la similaridad coseno con todos los documentos de train

In [ ]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [ ]:
np.sort(cossim)[::-1]

Después vemos a qué documentos corresponden

In [ ]:
np.argsort(cossim)[::-1]

Obtenemos los 5 documentos más similares:

In [ ]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

El documento original pertenece a la clase:

In [ ]:
newsgroups_train.target_names[y_train[idx]]

Revisamos las clases de los 5 más similares:

In [ ]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [ ]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [ ]:
f1_score(y_test, y_pred, average='macro')

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


# 1. Vectorizar documentos

In [32]:
import random
random.seed(131313)

# 5 indices random para samplear los documentos
random_indices = random.sample(range(X_train.shape[0]), 5)


random.seed(131313)

# 5 indices random para samplear los documentos
random_indices = random.sample(range(X_train.shape[0]), 5)

for i, idx in enumerate(random_indices, 1):
    print(f"\n\n\n {'='*20} Documento {i}: {idx} {'='*20}")

    # similaridad coseno del documento actual con todos los de train
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    
    # indices de los 5 documentos más similares
    mostsim_indices = np.argsort(cossim)[::-1][1:6]
    
    # etiqueta de la clase del documento original
    clase_original = newsgroups_train.target_names[y_train[idx]]
    
    print(f"Documento Original (Índice {idx}) - Clase: {clase_original}")
    print(f"Primeros 1000 caracteres: {newsgroups_train.data[idx][:1000].replace(chr(10), ' ')}...")
    print("Top 5 documentos más similares:")
    
    for rank, sim_idx in enumerate(mostsim_indices, 1):
        clase_sim = newsgroups_train.target_names[y_train[sim_idx]]
        score_sim = cossim[sim_idx]
        print(f"\n  {rank}. [Sim: {score_sim:.4f}] Clase: {clase_sim}")
        print(f"Primeros 1000 caracteres: {newsgroups_train.data[sim_idx][:1000].replace(chr(10), ' ')}...")
    print("-" * 50)





 ==================== Documento 1: 10615 ====================
Documento Original (Índice 10615) - Clase: misc.forsale
Primeros 1000 caracteres: I participated in a promotion by a company called Visual Images. I attempted to cancel my order before the package arrived. I was not able to stop them and now I have a package which I do not need.  Nishika 3D camera, wide angle flesh, film, carring case, instruction tapes, and some jewelrys.  3 vacation vouchers to Bahama, Cancun, Las Vegas, Orlando.  I paid $697 for the promotion package, and the vacation vouchers came as gift. I really want to sell them, so make me an offer for the whole package. If you are participating in a award, $697 is how much you would end up paying. And I strongly believe that you would get the same award as I do. If you are interested in those items, you could get them from me for a cheaper price.  Let me know, and make me an offer. No flames please, I have got enough.  You could reach me at koutd@hirama.hiram.ed

### Recordemos que el TF-IDF es una representación del tipo "bolsa de palabras", se basa exclusivamente en la frecuencia de los términos e ignora el orden y el contexto en el que aparecen. Como se ve esto en los resultados:

### 1. En el primer documento (10615, misc.forsale) tiene buena coherencia con sus dos más similares,  misma etiqueta y tema ("Visual Images", "promotion"). Estos dos comparten vocabulario especifico, incluso algunas frases identicas ("I attempted to cancel my order"). Sin embargo, a partir del tercero ya no tiene nada que ver, saltando a clases totalmente disímiles como talk.politics.misc o comp.graphics.

### 2. El segundo documento (6816, comp.graphics) no tiene coherencia con ninguno de sus similares. Relaciona un procesador "ARM" con un brazo anatómico (sci.med) o de suspensión (rec.autos), fallando completamente en la etiqueta porque el modelo ignora el contexto de la palabra.

### 3. El tercer documento (8489, comp.os.ms-windows.misc) tiene coherencia con su más similar (comparten clase y hablan de Windows NT), pero el segundo ya no tiene nada que ver, saltando abruptamente a un comunicado de la Casa Blanca en talk.politics.misc.

### 4. El cuarto documento (10493, rec.motorcycles) no tiene coherencia ni con el primero. El modelo lo asocia con etiquetas completamente disímiles como talk.religion.misc o soc.religion.christian.

### 5. El quinto documento (7844, rec.autos) tiene coherencia con su más similar (ambos son de autos y hablan de Volvo). Pero a partir del segundo se asocia erróneamente con etiquetas nada que ver como talk.politics.mideast y religión.

## 2. Construir un modelo de clasificación por prototipos (tipo zero-shot).


In [33]:
# calculamos la matriz de similaridades entre todos los de test y todos los de train
#  tamaño (n_test, n_train)
cossim_matrix = cosine_similarity(X_test, X_train)

# para cada fila (test), buscamos la columna (train) con mayor similaridad
most_similar_indices = np.argmax(cossim_matrix, axis=1)

y_pred_prototipo = y_train[most_similar_indices]

# evaluamos usando F1-Score Macro
f1_macro_prototipo = f1_score(y_test, y_pred_prototipo, average='macro')

print(f"F1-Score Macro (Clasificador por Prototipo): {f1_macro_prototipo:.4f}")

F1-Score Macro (Clasificador por Prototipo): 0.5050


### Este modelo arroja un desempeño base aceptable pero sufre en la generalización. Al depender exclusivamente del vecino más cercano en el conjunto de entrenamiento es extremadamente sensible a la polisemia y al peso desproporcionado de ciertas palabras. Si un texto de prueba comparte un término infrecuente pero descontextualizado con otro de entrenamiento (como nos pasó con "ARM" en el punto anterior), la similitud se dispara y la predicción es pobre. A diferencia de un modelo probabilístico, este enfoque no "aprende" las características generales del vocabulario de una clase, se ve castigado por coincidencias exactas de palabras infrecuentes, sin noci[on de contexto.

## 3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación

In [34]:
from sklearn.pipeline import Pipeline


experimentos = [
    ('MultinomialNB_default', TfidfVectorizer(), MultinomialNB()),
    ('MultinomialNB_alpha0.1', TfidfVectorizer(stop_words='english', min_df=2), MultinomialNB(alpha=0.1)),
    ('ComplementNB_default', TfidfVectorizer(), ComplementNB()),
    ('ComplementNB_optimo', TfidfVectorizer(stop_words='english', sublinear_tf=True), ComplementNB(alpha=0.1))
]

for nombre, vect, model in experimentos:
    pipeline = Pipeline([('vect', vect), ('clf', model)])
    pipeline.fit(newsgroups_train.data, y_train)
    y_pred = pipeline.predict(newsgroups_test.data)
    score = f1_score(y_test, y_pred, average='macro')
    print(f"{nombre}: F1-Score Macro = {score:.4f}")

MultinomialNB_default: F1-Score Macro = 0.5854
MultinomialNB_alpha0.1: F1-Score Macro = 0.6798
ComplementNB_default: F1-Score Macro = 0.6930
ComplementNB_optimo: F1-Score Macro = 0.6900


### El modelo Naive Bayes al ser tipo BoW, asume la independencia entre términos y se basa en el cálculo de probabilidades por categoría. Los resultados muestran que el f1-macro mejora mucho al ajustar el parámetro de suavizado $\alpha$ y filtrar stop words, ya que esto permite que el modelo se enfoque en las palabras "interesantes".
### Por último, observamos que ComplementNB supera al MultinomialNB estándar (0.6930 vs 0.5854), lo cual es coherente con la teoría, ya que esta variante está diseñada específicamente para ser más robusta cuando existe un desbalance en las categorías del corpus.

## 4. Transponer la matriz documento-término.

In [35]:
X_words = X_train.T

X_words.data

array([0.20858239, 0.11904551, 0.08965702, ..., 0.24129371, 0.24129371,
       0.24129371], shape=(1103627,))

In [ ]:
vocabulario_completo = list(tfidfvect.vocabulary_.keys())
palabras_azar = random.sample(vocabulario_completo, 10)
# voy viendo las palabras al azar hasta ver 5 que me gusten, en general son horribles
palabras_azar

# correction, surprise, scholar, renew, colourful
# voy a inventar porque esto esta tomando mucho tiempo y muestra mucha basura
# elijo: space, health, tennis, god, rock

['backbite',
 'c9khl',
 'kzyra',
 'dilly',
 'heretical',
 'skirmishes',
 'eu',
 'one',
 'slqk',
 '10x20']

In [85]:
# palabras_interesantes = ['correction', 'surprise', 'scholar', 'renew', 'colourful'] # dan pesimos resultados
palabras_interesantes = ['space', 'health', 'tennis', 'god', 'rock']

for palabra in palabras_interesantes:
    if palabra in tfidfvect.vocabulary_:
        idx_palabra = tfidfvect.vocabulary_[palabra]
        
        #  en cuántos documentos aparece la palabra principal
        docs_palabra = X_words[idx_palabra].nnz
        
        sim_palabras = cosine_similarity(X_words[idx_palabra], X_words)[0]
        
        indices_sim = np.argsort(sim_palabras)[::-1][1:6]
        
        print(f"Palabra: '{palabra}' (Aparece en {docs_palabra} docs)")
        for i in indices_sim:
            docs_sim = X_words[i].nnz
            print(f"  -> {idx2word[i]} (Sim: {sim_palabras[i]:.4f} | Docs: {docs_sim})")
        print("-" * 50)

Palabra: 'space' (Aparece en 398 docs)
  -> nasa (Sim: 0.3304 | Docs: 157)
  -> seds (Sim: 0.2966 | Docs: 3)
  -> shuttle (Sim: 0.2928 | Docs: 71)
  -> enfant (Sim: 0.2803 | Docs: 4)
  -> seti (Sim: 0.2465 | Docs: 4)
--------------------------------------------------
Palabra: 'health' (Aparece en 122 docs)
  -> ohip (Sim: 0.3304 | Docs: 4)
  -> provincial (Sim: 0.2998 | Docs: 4)
  -> care (Sim: 0.2824 | Docs: 301)
  -> breakoff (Sim: 0.2799 | Docs: 2)
  -> traditionalists (Sim: 0.2799 | Docs: 2)
--------------------------------------------------
Palabra: 'tennis' (Aparece en 11 docs)
  -> inundate (Sim: 0.5842 | Docs: 1)
  -> victums (Sim: 0.5842 | Docs: 1)
  -> licnese (Sim: 0.5842 | Docs: 1)
  -> triggers (Sim: 0.5620 | Docs: 2)
  -> dehydrated (Sim: 0.4652 | Docs: 2)
--------------------------------------------------
Palabra: 'god' (Aparece en 579 docs)
  -> jesus (Sim: 0.2688 | Docs: 263)
  -> bible (Sim: 0.2616 | Docs: 243)
  -> that (Sim: 0.2560 | Docs: 6580)
  -> existence (Sim:

### Al transponer la matriz para evaluar la coocurrencia de palabras. Los resultados muestran que los términos con alta frecuencia logran agrupaciones semánticas coherentes (como god [579 docs] con jesus [263 docs]). Sin embargo, la  sparsidad del modelo provoca que términos poco frecuentes como tennis se asocien con faltas de ortografía como licnese (1 doc) por el simple hecho de coincidir de forma aislada en un único texto. Finalmente, la intrusión de that (6580 docs) como vecina de god evidencia cómo las stop words ensucian el espacio vectorial, demostrando que sin un preprocesamiento riguroso (eliminación de palabras vacías y umbrales de frecuencia mínima),  el ruido estadístico domina la similitud. 

### Probemos ahora limpiando un poco los docs:

In [87]:
tfidfvect_clean = TfidfVectorizer(stop_words='english', min_df=5)
X_train_clean = tfidfvect_clean.fit_transform(newsgroups_train.data)
X_words_clean = X_train_clean.T

palabras_interesantes = ['space', 'health', 'tennis', 'god', 'rock']

for palabra in palabras_interesantes:
    if palabra in tfidfvect_clean.vocabulary_:
        idx_palabra = tfidfvect_clean.vocabulary_[palabra]
        docs_palabra = X_words_clean[idx_palabra].nnz
        sim_palabras = cosine_similarity(X_words_clean[idx_palabra], X_words_clean)[0]
        indices_sim = np.argsort(sim_palabras)[::-1][1:6]
        
        print(f"Palabra: '{palabra}' (Aparece en {docs_palabra} docs)")
        for i in indices_sim:
            # Obtenemos la palabra limpia desde el nuevo vocabulario
            idx2word_clean = {v: k for k, v in tfidfvect_clean.vocabulary_.items()}
            docs_sim = X_words_clean[i].nnz
            print(f"  -> {idx2word_clean[i]} (Sim: {sim_palabras[i]:.4f} | Docs: {docs_sim})")
        print("-" * 50)

Palabra: 'space' (Aparece en 398 docs)
  -> nasa (Sim: 0.3178 | Docs: 157)
  -> shuttle (Sim: 0.2784 | Docs: 71)
  -> exploration (Sim: 0.2328 | Docs: 30)
  -> aeronautics (Sim: 0.2219 | Docs: 12)
  -> cfa (Sim: 0.2164 | Docs: 5)
--------------------------------------------------
Palabra: 'health' (Aparece en 122 docs)
  -> care (Sim: 0.2809 | Docs: 301)
  -> insurance (Sim: 0.2672 | Docs: 92)
  -> premiums (Sim: 0.2213 | Docs: 7)
  -> resent (Sim: 0.1969 | Docs: 9)
  -> allready (Sim: 0.1830 | Docs: 5)
--------------------------------------------------
Palabra: 'tennis' (Aparece en 11 docs)
  -> shoes (Sim: 0.4107 | Docs: 15)
  -> buildup (Sim: 0.3577 | Docs: 5)
  -> migraine (Sim: 0.2819 | Docs: 11)
  -> 2500 (Sim: 0.2388 | Docs: 10)
  -> sensation (Sim: 0.2350 | Docs: 6)
--------------------------------------------------
Palabra: 'god' (Aparece en 579 docs)
  -> jesus (Sim: 0.2768 | Docs: 263)
  -> bible (Sim: 0.2675 | Docs: 243)
  -> christ (Sim: 0.2674 | Docs: 197)
  -> faith (Sim

### Al aplicar el preprocesamiento de texto (como la eliminación de stop words y un umbral de frecuencia mínima), la calidad de las representaciones vectoriales mejora significativamente. Por un lado, excluir palabras comunes que no aportan valor semántico  elimina el ruido estadístico; esto se observa claramente al desaparecer that del entorno de god, dando lugar a palabras exclusivamente de la temática (christ, faith). Por otro lado, el filtro de frecuencia mitiga la por esparsidad: tennis ya no se asocia a errores ortográficos de un solo documento, sino a co-ocurrencias más lógicas como shoes, mientras que health consolida un grupo perfectamenmte coherente (care, insurance, premiums). Estos resultados validan que, en representaciones tipo BoW, la limpieza de los datos es clave.